<a href="https://colab.research.google.com/github/wushuang1005/llm-demo/blob/main/nanoGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-09 07:15:49--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-03-09 07:15:50 (33.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
with open("input.txt", "r") as f:
    text = f.read()

print(f"总字符数: {len(text)}")
print(text[:200])

# 字符级别的tokenizer（最简单的方式）
chars = sorted(set(text))
vocab_size = len(chars)
print(f"词表大小: {vocab_size} 个字符")

# 字符 <-> 数字 的映射
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

print(encode("hello"))
print(decode(encode("hello")))

总字符数: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you
词表大小: 65 个字符
[46, 43, 50, 50, 53]
hello


In [3]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"训练集: {len(train_data)} tokens")
print(f"验证集: {len(val_data)} tokens")

训练集: 1003854 tokens
验证集: 111540 tokens


In [4]:
block_size = 128   # 每次看多长的上下文
batch_size = 32    # 每批多少条
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用设备: {device}")

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i: i+block_size] for i in ix])
    y = torch.stack([data[i+1: i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# x是输入，y是x往右移一位（预测下一个词）
x, y = get_batch("train")
print(f"输入shape: {x.shape}")   # [32, 128]
print(f"标签shape: {y.shape}")   # [32, 128]

使用设备: cuda
输入shape: torch.Size([32, 128])
标签shape: torch.Size([32, 128])


In [5]:
import torch.nn as nn
import torch.nn.functional as F

# 超参数
n_embd = 128      # 词向量维度
n_head = 4        # 多头注意力的头数
n_layer = 4       # Transformer Block的层数
dropout = 0.1

class Head(nn.Module):
    """单个注意力头"""
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # 因果掩码：不能看未来的词
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # Q × K^T / sqrt(head_size) → attention分数
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))  # 遮住未来
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        return wei @ v  # 加权求和V

class MultiHeadAttention(nn.Module):
    """多头注意力：并行跑多个Head，再拼接"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    """每个位置独立的两层全连接"""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """一个完整的Transformer Block"""
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)  # 自注意力
        self.ff   = FeedForward(n_embd)                    # 前馈网络
        self.ln1  = nn.LayerNorm(n_embd)                   # 归一化
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))   # 残差连接
        x = x + self.ff(self.ln2(x))   # 残差连接
        return x

class NanoGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f   = nn.LayerNorm(n_embd)
        self.head   = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)                              # (B,T,C)
        pos_emb = self.position_embedding(torch.arange(T, device=device))# (T,C)
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)  # (B,T,vocab_size)

        if targets is None:
            return logits, None

        # 计算loss
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs  = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx


model = NanoGPT().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {total_params/1e6:.2f}M")

模型参数量: 0.82M


In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
eval_interval = 200
max_iters = 3000

@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}
    for split in ["train", "val"]:
        L = []
        for _ in range(50):
            x, y = get_batch(split)
            _, loss = model(x, y)
            L.append(loss.item())
        losses[split] = sum(L) / len(L)
    model.train()
    return losses

for i in range(max_iters):
    if i % eval_interval == 0:
        losses = estimate_loss()
        print(f"Step {i}: train loss={losses['train']:.4f}, val loss={losses['val']:.4f}")

    x, y = get_batch("train")
    logits, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("训练完成！")

Step 0: train loss=4.2910, val loss=4.2970
Step 200: train loss=2.5222, val loss=2.5363
Step 400: train loss=2.4375, val loss=2.4454
Step 600: train loss=2.3562, val loss=2.3630
Step 800: train loss=2.2468, val loss=2.2736
Step 1000: train loss=2.1482, val loss=2.1824
Step 1200: train loss=2.0671, val loss=2.0996
Step 1400: train loss=1.9921, val loss=2.0532
Step 1600: train loss=1.9363, val loss=2.0082
Step 1800: train loss=1.8846, val loss=1.9722
Step 2000: train loss=1.8339, val loss=1.9435
Step 2200: train loss=1.7919, val loss=1.9154
Step 2400: train loss=1.7537, val loss=1.8916
Step 2600: train loss=1.7258, val loss=1.8552
Step 2800: train loss=1.6967, val loss=1.8429
训练完成！


In [7]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=300)
print(decode(generated[0].tolist()))


Wjelse wilThering hom madon. Thy not:
Thou sukesly well that manirang to knoor?

Good good:
You yourd hour to, that, thy Haps cover
julettress: with alr I seake ea, was anl
Oie lops an braw not-I warraword: as thre
gignter ctrumb thour perecerof, I with teak

'TAUCKII:
Siver'd very folmes herd-pos p


In [8]:
import types

def new_generate(self, idx, max_new_tokens, temperature=1.0):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = self(idx_cond)
        logits = logits[:, -1, :] / temperature
        probs  = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, idx_next], dim=1)
    return idx

# 直接替换已有model的generate方法，不影响训练好的权重
model.generate = types.MethodType(new_generate, model)
print("generate方法已更新！")

generate方法已更新！


In [9]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

for temp in [0.5, 1.0, 1.5]:
    generated = model.generate(context, max_new_tokens=150, temperature=temp)
    print(f"\n===== temperature={temp} =====")
    print(decode(generated[0].tolist()))


===== temperature=0.5 =====

Nur to the world in here wash the sell in will
There nore who seech more the coust a better,
Whe are well the seep the thou her ander and
That and the

===== temperature=1.0 =====

I'll would woot, noth and keeman:
Offear, bady me underim, work lafe,--my your souns,
As mord no dist do erough: That all andwrab the she;
And Sut bet

===== temperature=1.5 =====

But is Lencerrees goighed;-amot, untalk Tis Valiforce; a chbe
on leart, ot, Led: God, parcubxe.

MENIUS:
SisM But! is;
In, in:
parecing.

Ko Exaventom


In [10]:
from google.colab import drive
drive.mount('/content/drive')

# 保存到你的Drive
torch.save(model.state_dict(), "/content/drive/MyDrive/nanogpt_weights.pt")

Mounted at /content/drive


In [ ]:
drive.mount('/content/drive')

model = NanoGPT().to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/nanogpt_weights.pt"))
model.eval()
print("模型加载成功！")